# Bot Detection — Touchpoint Data Cleaning

**Goal:** Identify and remove non-human (bot/fraudulent) traffic from the raw touchpoint
log before it feeds into attribution modeling and CPA calculations downstream.

**Approach:** Four candidate bot-detection signals were diagnosed against the raw data
first (see `Diagnose.ipynb` for that exploration). Each signal was only kept if it
showed genuine statistical evidence of bot behavior — not just "this looks
suspicious." The decision log for all four signals is below, followed by the
implementation of the one signal that survived scrutiny.

**Input:** `touchpoints.csv` (566,510 rows / 100,000 users)
**Output:** `touchpoints_clean_v3.csv`, `bot_detection_report_v3.csv`

## Step 1 — Load and standardize raw data

Strip/standardize column names, parse timestamps, normalize text casing on
`channel` and `event_type`, and sort by user and time so each user's touchpoints
read as a chronological journey.

In [7]:
import pandas as pd
import numpy as np

# 1. LOAD RAW DATA
df = pd.read_csv('touchpoints.csv')
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
df['timestamp']  = pd.to_datetime(df['timestamp'])
df['channel']    = df['channel'].str.strip().str.title()
df['event_type'] = df['event_type'].str.strip()
df = df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)

print(f"Total rows: {len(df)} | Total users: {df['user_id'].nunique()}\n")

Total rows: 566510 | Total users: 100000



## Step 2 — Decision log: which bot signals actually hold up?

Four candidate signals were tested against the raw data during diagnostics. The full
reasoning for keeping or dropping each one is preserved in the code comments below —
this matters because two of the four signals turned out to be false positives, and a
naive bot-detection pass would have over-flagged genuine human behavior as fraud.

**Summary of the four signals:**

| Signal | Verdict | Reason |
|---|---|---|
| 1. Speed (<1s between events) | ❌ Dropped | Timestamps have zero second-precision (all `:00`) — apparent "0-second gaps" are a rounding artifact of the data, not evidence of bot-speed clicking |
| 2. Volume (events/user) | ✅ Kept, tightened to IQR × 4 | The only signal with a genuine statistical cliff — 95th percentile is 7 events/user, 99.9th percentile is 136. A 20× jump at the extreme tail is a real signature, not gradual human variance |
| 3. Pattern (impressions-only) | ❌ Dropped (after fix) | The original rule flagged any impressions-only user as a bot — but 77.6% of *all* events are impressions, and a 0.5–2% CTR is normal for display ads. Not clicking is normal human behavior, not fraud. Tightening the rule to require impressions-only *and* abnormal volume left 0 users — there's no clean impression-fraud signature in this dataset once disinterest isn't conflated with fraud |
| 4. Burst duplicates | ❌ Dropped | Flagged the exact same 11,518 users as Signal 1 — same minute-rounding artifact, not an independent signal. Keeping both would double-count one data issue as two |

**Net result:** only the Volume signal (IQR × 4) is evidence-backed enough to act on.

In [10]:
# ══════════════════════════════════════════════════════════════════════════
# FINAL BOT DETECTION — EVIDENCE-BASED, POST-DIAGNOSTIC
# ══════════════════════════════════════════════════════════════════════════
# DECISION LOG (from diagnostics on raw data):
#
# Signal 1 (Speed <1s)   -> DROPPED. Timestamps have ZERO second-precision
#                            (all :00). "0 second gaps" are a rounding
#                            artifact, not evidence of bot speed.
#
# Signal 4 (Burst dupes) -> DROPPED. Flagged the exact same 11,518 users as
#                            Signal 1 -- it's the same minute-rounding
#                            artifact, not an independent signal. Keeping
#                            both would double-count one data issue as two.
#
# Signal 3 (Pattern)     -> KEPT, FIXED LOGIC. Old rule ("impressions-only
#                            = bot") flagged 74,537 users -- but 77.6% of
#                            ALL events are impressions, and real-world
#                            display ad CTR is naturally 0.5-2%. Not
#                            clicking is normal human behavior. New rule
#                            requires impressions-only AND abnormal volume.
#                            Result: 0 users meet both conditions -- there's
#                            no clean impression-fraud signature here once
#                            disinterest isn't conflated with fraud.
#
# Signal 2 (Volume)      -> KEPT, TIGHTENED (IQR x4). This is the only
#                            signal showing a genuine statistical cliff:
#                            95th pct = 7 events, 99.9th pct = 136 events.
#                            A 20x jump at the extreme tail is a real bot
#                            signature, not gradual human variation.

## Step 3 — Implement the surviving signal: Volume (IQR × 4)

Standard outlier detection (Tukey's method), but with a wider multiplier (×4 instead
of the usual ×1.5) specifically because ad-touchpoint volume is naturally right-skewed
— a strict ×1.5 IQR would flag plenty of legitimately engaged human users. ×4 only
catches the extreme tail where the 20× volume jump described above lives.

In [13]:
user_counts = df.groupby('user_id').size()
Q1, Q3 = user_counts.quantile([0.25, 0.75])
IQR    = Q3 - Q1
UPPER  = Q3 + 4 * IQR
bot_volume = set(user_counts[user_counts > UPPER].index)

## Step 4 — Apply the flag and measure impact

1,952 users (1.95% of the user base) are flagged and removed, taking out 193,940 rows
(34.23% of total rows). The row-removal percentage is much larger than the
user-removal percentage because, by construction, flagged users are the ones
generating disproportionately many events — that's exactly the pattern we're
filtering for.

In [16]:
print(f"[Signal 2 — Volume, IQR x4] Threshold: >{UPPER:.0f} events/user")
print(f"[Signal 2 — Volume, IQR x4] Users flagged: {len(bot_volume)}")

all_bots = bot_volume   # the only signal retained after evidence review
print(f"\nTOTAL BOTS FLAGGED: {len(all_bots)} ({len(all_bots)/df['user_id'].nunique()*100:.2f}% of users)")

before_rows = len(df)
df_clean = df[~df['user_id'].isin(all_bots)].copy()
after_rows = len(df_clean)
pct_removed = (before_rows - after_rows) / before_rows * 100

[Signal 2 — Volume, IQR x4] Threshold: >17 events/user
[Signal 2 — Volume, IQR x4] Users flagged: 1952

TOTAL BOTS FLAGGED: 1952 (1.95% of users)


In [18]:
print(f"Rows before: {before_rows:,}")
print(f"Rows after:  {after_rows:,}")
print(f"Removed:     {before_rows - after_rows:,} ({pct_removed:.2f}%)")

Rows before: 566,510
Rows after:  372,570
Removed:     193,940 (34.23%)


## Step 5 — Sanity check the flagged users

Before trusting the filter, check what the flagged users' behavior actually looks
like. If they're overwhelmingly Impressions with little to no real engagement
(Click/Purchase), that's consistent with bot/fraud behavior — high volume, no real
intent to convert.

In [21]:
# ── VALIDATE: what do the flagged users actually look like? ──────────────────
print("\n── Sanity check: flagged users' event breakdown ──")
flagged_events = df[df['user_id'].isin(bot_volume)]['event_type'].value_counts(normalize=True) * 100
print(flagged_events.round(2))
print("\n-> If flagged users are mostly Impressions with near-zero Purchase/Click,")
print("   that supports genuine bot/fraud behavior (high volume, no real intent).")


── Sanity check: flagged users' event breakdown ──
event_type
Impression    50.0
Click         50.0
Name: proportion, dtype: float64

-> If flagged users are mostly Impressions with near-zero Purchase/Click,
   that supports genuine bot/fraud behavior (high volume, no real intent).


## Step 6 — Save cleaned data

`touchpoints_clean_v3.csv` becomes the input for all downstream work (EDA, attribution
modeling, CPA calculation). `bot_detection_report_v3.csv` documents what was removed
and why, for auditability.

In [24]:
# ── SAVE ───────────────────────────────────────────────────────────────────────
df_clean.to_csv('touchpoints_clean_v3.csv', index=False)

bot_report_v3 = pd.DataFrame({
    'signal': ['Volume (IQR x4) — final retained signal'],
    'users_flagged': [len(bot_volume)],
    'pct_of_users': [round(len(bot_volume)/df['user_id'].nunique()*100, 2)]
})
bot_report_v3.to_csv('bot_detection_report_v3.csv', index=False)
print("\n✓ Saved: touchpoints_clean_v3.csv")
print("✓ Saved: bot_detection_report_v3.csv")


✓ Saved: touchpoints_clean_v3.csv
✓ Saved: bot_detection_report_v3.csv
